# BR01B — Image-only MIMIC-CXR candidates

MedGemma CUDA environment. Uses native reference images, no reports/labels, and records extractor revision and image hashes. All eligible candidates must succeed before index finalization. Extraction is checkpointed per image.

In [ ]:
from pathlib import Path
import sys
EXP = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / 'config.example.json').is_file() and (p / 'common.py').is_file()), None)
if EXP is None: raise FileNotFoundError('Start Jupyter inside graph_reasoning/experiment02')
sys.path.insert(0, str(EXP))
from common import load_config, parent_setup, read
CONFIG_PATH = EXP / 'config.json'
if not CONFIG_PATH.is_file():
    raise FileNotFoundError('Run BR00 first to create experiment02/config.json')
CFG = load_config(CONFIG_PATH)
PROJECT = parent_setup(CFG)
RUN = Path(CFG['run_dir'])
print('Kernel:', sys.executable)
print('Cohort:', CFG['cohort'], '| outputs:', RUN)


In [ ]:
from common import parent_setup
from ecg_external_generation import local_snapshot
from sensor_candidate_generation import load_runtime
MODEL_PATH = None
MODEL_PATH = local_snapshot('google/medgemma-1.5-4b-it', MODEL_PATH)
runtime = load_runtime('cxr', MODEL_PATH)
print('MedGemma revision:', runtime['revision'])


In [ ]:
from encoders import extract_cxr
MAX_NEW = 3  # Inspect pilot, then 0 for all remaining.
RETRY_ERRORS = False
MAX_RUN_HOURS = 30.0
MAX_NEW_TOKENS = 1024  # Frozen per experiment; changed settings require a new run_dir.
print(extract_cxr(CFG, runtime, max_new=MAX_NEW, retry_errors=RETRY_ERRORS,
                  max_run_hours=MAX_RUN_HOURS, max_new_tokens=MAX_NEW_TOKENS))
